# ML-03 — Frame My Lane as an ML Task

**Lane:** Refresh / Content Opportunity Scoring. This notebook frames a decision-support queue for a human content reviewer; it does not automate page changes.

## 1. My lane as an ML task (type)

This is a **ranking / scoring** task. One score is assigned to each existing content page, then the pages are ordered so a content or SEO reviewer can inspect the highest-priority pages first. The decision is *which 50 pages should enter this review queue now?* The reviewer then chooses whether to refresh, expand, monitor, protect, or skip each page.

A wrong high score wastes limited editor time; a wrong low score can leave a high-demand weak page unseen. The output is decision support, not a claim that editing a page will cause recovery.

In [1]:
import os
from pathlib import Path

import pandas as pd

# Make the notebook work whether it starts at the repository root or in work/notebooks.
start = Path.cwd()
root = next(
    candidate for candidate in [start, *start.parents]
    if (candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists()
)
os.chdir(root)

csv_path = Path('data/raw/content_refresh_anonymized.csv')
df = pd.read_csv(csv_path)

print(f'Loaded {len(df):,} content pages from {csv_path}')
print(f'Clients represented: {df.client_id.nunique()}')

Loaded 30,000 content pages from data\raw\content_refresh_anonymized.csv
Clients represented: 32


## 2. Target or proxy

The target I ultimately want is an **observed future outcome**: `future_decline_30d = 1` when a page's search impressions in a later 30-day window fall at least 20% from its preceding window, and `0` otherwise. That label will be built from later daily warehouse data, after the feature window ends.

The starter CSV has no separate future window. For this framing exercise only, I sketch `decline_proxy_same_window` from `trend_direction == 'down'`. It is a defined, same-window proxy—not a future-observed label—and neither `trend_direction` nor `trend_pct` may be used as features when learning from it.

In [2]:
# Sketch of the available starter proxy. It is deliberately named as a proxy.
df['decline_proxy_same_window'] = (df['trend_direction'] == 'down').astype('int8')

proxy_counts = df['decline_proxy_same_window'].value_counts().sort_index()
print('Target column sketch: decline_proxy_same_window (0 = not down, 1 = down)')
print(proxy_counts.rename(index={0: '0: not down', 1: '1: down'}).to_string())
print(f"Proxy positive rate: {df['decline_proxy_same_window'].mean():.1%}")
print('Later target: future_decline_30d, measured after the feature window.')
print('Leakage guard: trend_direction and trend_pct are label sources, never predictive features.')

Target column sketch: decline_proxy_same_window (0 = not down, 1 = down)
decline_proxy_same_window
0: not down    13738
1: down        16262
Proxy positive rate: 54.2%
Later target: future_decline_30d, measured after the feature window.
Leakage guard: trend_direction and trend_pct are label sources, never predictive features.


## 3. Success metric

My primary metric is **precision@50**: among the 50 pages at the top of the queue, the fraction with `future_decline_30d = 1`. It matches the real capacity decision—reviewing 50 pages—not average performance across every page.

A provisional success threshold is precision@50 of **at least 65%** on a held-out set of clients, and at least 10 percentage points above a transparent rule baseline. I will report the result with the proxy clearly labelled on the starter slice, then validate the same metric with a future-window label later.

In [3]:
review_budget = 50
proxy_rate = df['decline_proxy_same_window'].mean()

print(f'Review capacity (K): {review_budget} pages')
print(f'Starter proxy prevalence: {proxy_rate:.1%}')
print('Pre-registered later success check: precision@50 >= 65% and >= baseline + 10 percentage points.')

Review capacity (K): 50 pages
Starter proxy prevalence: 54.2%
Pre-registered later success check: precision@50 >= 65% and >= baseline + 10 percentage points.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (page)** measured over the starter dataset's trailing 90-day window. `content_id` is unique and is used only to verify this grain; it is intentionally not a model feature and is not displayed below. The displayed columns are page-level signals and the clearly labelled proxy target.

In [4]:
# Grain checks before showing the page-level dataframe.
assert df['content_id'].is_unique, 'Expected one row per content item.'
assert len(df) == 30_000, 'Unexpected starter-slice row count.'

unit_of_analysis = df[[
    'content_type', 'content_age_days', 'days_since_last_update',
    'impressions_90d', 'ctr', 'avg_position', 'engagement_rate',
    'decline_proxy_same_window'
]].copy()

print(f'Grain check: {len(df):,} rows and {df.content_id.nunique():,} unique content ids.')
print('One row below = one content item/page; no client names, URLs, or queries are shown.')
unit_of_analysis.head(8)

Grain check: 30,000 rows and 30,000 unique content ids.
One row below = one content item/page; no client names, URLs, or queries are shown.


,content_type,content_age_days,days_since_last_update,impressions_90d,ctr,avg_position,engagement_rate,decline_proxy_same_window
0,keyword article,187,20,3803,0.76,10.6,5.88,1
1,keyword article,445,25,15320,0.05,20.3,0.00,1
2,keyword article,141,20,12581,0.09,36.5,0.00,1
3,keyword article,463,22,11751,0.49,6.2,1.28,0
4,keyword article,263,14,19140,0.13,44.0,0.00,1
5,keyword article,147,20,3970,0.03,8.5,0.00,1
6,keyword article,90,20,20,0.00,7.0,0.00,1
7,keyword article,445,22,1724,0.06,21.2,3.57,0


## 5. Why ML beats a fixed rule here

A fixed rule is still my baseline: for example, flag a stale page with meaningful search demand. But it cannot sensibly rank the many trade-offs in this data. A recently updated page with very high impressions and weak CTR may deserve review before an old page with little demand; a page with a poor position may have a different action from a page with a strong position but a falling trend. Freshness, demand, CTR, position, engagement, and content type interact rather than sharing one defensible threshold.

ML earns its place only if a leakage-safe score ranks the top 50 better than that transparent rule. If it does not, I should keep the rule. The score supports a human review action; it does not replace judgement or automatically make content changes.

In [5]:
# A simple rule demonstrates why a binary flag is not yet a useful queue.
stale_visible_rule = (
    (df['days_since_last_update'] >= 180)
    & (df['impressions_90d'] >= 500)
)
high_demand_proxy = (
    (df['impressions_90d'] >= 100)
    & (df['decline_proxy_same_window'] == 1)
)

print(f"Simple 'stale and visible' rule flags: {int(stale_visible_rule.sum()):,} pages")
print(f"Demand + same-window decline proxy: {int(high_demand_proxy.sum()):,} pages")
print(f'Review capacity: {review_budget} pages')
print('A useful queue must order competing signals, not merely flag a large or tiny bucket.')

Simple 'stale and visible' rule flags: 17 pages
Demand + same-window decline proxy: 13,152 pages
Review capacity: 50 pages
A useful queue must order competing signals, not merely flag a large or tiny bucket.


## Self-check

- [x] I named a ranking/scoring task, target/proxy, action, and success metric.
- [x] I showed the actual page-level unit of analysis from the starter slice.
- [x] I explained why a fixed rule is a baseline, not automatically the final answer.
- [x] I labelled the starter outcome as a same-window proxy and excluded its label sources from features.
- [x] The notebook is executed top to bottom before submission.